# Deep Q-Learning Trading Agent (DQN): AAPL

A DQN agent learns to **buy / sell / hold** AAPL from the technical features in `AAPL_trading_features.csv` (Close, SMA_20, SMA_50, Volatility, BB_upper, BB_lower, RSI). It trains on the first 70% of dates, is evaluated on the remaining 30%, and is compared with buy-and-hold and a random policy. **The agent, episode runner, reports and charts all live in `func_lib.py`.**

### Why the earlier backtest was empty, and what changed
The earlier version trained an agent that **bought every day and never sold**: 335 open positions and *zero closed trades* in the test period. So the backtest, trade log and trade charts had nothing to show. Four design issues caused it, and each is fixed here:

| Problem | Fix |
|---|---|
| The state had no inventory information, so the agent couldn't tell whether a sell was possible. The greedy policy collapsed to a single action. | The state now includes **position features**: share of max inventory held, a holding flag, and unrealized return. |
| Reward came only from profitable sells (`max(profit, 0)`), so losing positions and idle days looked identical. | **Mark-to-market reward:** the daily P&L of the position, in % of price, net of `COST_BPS`. `REWARD_MODE='realized'` restores the original. |
| Epsilon decayed 0.995 per replay step, hitting its floor after ~900 steps, inside episode 1. | Decay is **scheduled** so epsilon reaches its minimum after 80% of all training steps. |
| Unlimited buying; open lots were never closed, so they never reached the trade log. | `MAX_INVENTORY` caps the position. Invalid actions are **masked** (no selling with nothing held, no buying when full). Lots still open on the last day are closed ("End of data"). |

A **target network** (`TARGET_UPDATE_EVERY`) stabilizes the Bellman targets. The bugs fixed earlier (date split, train-only scaler, batched random replay, name-based columns) are kept.

In [1]:
# Imports, plus the func_lib helpers for building states, running episodes and reporting trades.
# Seeds Python, NumPy and Keras so results repeat, and silences Keras progress bars.
import os
import sys

import keras
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Make func_lib importable whether the notebook runs from this folder or its parent
for path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))):
    if os.path.exists(os.path.join(path, 'func_lib.py')) and path not in sys.path:
        sys.path.append(path)

import func_lib   # importing it also applies the dark Plotly theme
from func_lib import (ACTIONS, N_POSITION_FEATURES, Agent, build_states, plot_behavior, plot_losses,
                      print_summary, print_trade_statistics, run_episode, trade_analysis_figure,
                      trade_report, trade_statistics)

SEED = 42
keras.utils.set_random_seed(SEED)          # seeds Python, NumPy and the Keras backend
keras.utils.disable_interactive_logging()  # silence per-call Keras progress bars

In [2]:
# ---- Configuration ----
# Data file and train/test split, training episodes, inventory cap, reward type and trading costs,
# plus the agent's learning parameters.
DATA_FILE = 'AAPL_trading_features.csv'
TRAIN_FRACTION = 0.7
NORMALIZE = True          # scale features with a StandardScaler fitted on TRAIN data only

WINDOW_SIZE = 1           # number of daily feature changes in each state
EPISODES = 5
BATCH_SIZE = 32
TRAIN_EVERY = 1           # run experience replay every N steps (raise to train faster)
MAX_INVENTORY = 5         # max shares held at once
REWARD_MODE = 'mtm'       # 'mtm' (daily mark-to-market, recommended) or 'realized' (original)
COST_BPS = 5              # transaction cost per share traded, basis points of price
VERBOSE_TRADES = False    # print every buy/sell

AGENT_PARAMS = dict(memory_size=2000, gamma=0.95, epsilon=1.0, epsilon_min=0.02,
                    learning_rate=1e-3, target_update_every=200)
RUN_PARAMS = dict(batch_size=BATCH_SIZE, train_every=TRAIN_EVERY, max_inventory=MAX_INVENTORY,
                  reward_mode=REWARD_MODE, cost_bps=COST_BPS, verbose=VERBOSE_TRADES)
FINAL_MODEL_PATH = f'model_ep{EPISODES}.keras'   # saved after the last training episode

## 1. Data and chronological split

In [3]:
# Load the seven trading features saved by the time-series notebook, drop empty rows, and split by
# date: the first 70% of days for training, the rest held back as an unseen test period.
data = pd.read_csv(DATA_FILE, parse_dates=['Date']).set_index('Date').sort_index()
data = data.loc[:, ~data.columns.str.startswith('Unnamed')]
print('Missing values per column:', data.isna().sum().to_dict())
data = data.dropna()

unique_dates = data.index.unique()
split_date = unique_dates[int(TRAIN_FRACTION * len(unique_dates))]
train_df = data[data.index < split_date]
test_df = data[data.index >= split_date]

FEATURES = list(data.columns)
print(f'{len(data):,} rows: {data.index[0]:%Y-%m-%d} to {data.index[-1]:%Y-%m-%d}')
print(f'Split date: {split_date:%Y-%m-%d} | train: {train_df.shape} | test: {test_df.shape}')
print('Features:', FEATURES)

Missing values per column: {'Close': 0, 'SMA_20': 0, 'SMA_50': 0, 'Volatility': 0, 'BB_upper': 0, 'BB_lower': 0, 'RSI': 0}
1,124 rows: 2020-03-13 to 2024-08-29
Split date: 2023-04-27 | train: (786, 7) | test: (338, 7)
Features: ['Close', 'SMA_20', 'SMA_50', 'Volatility', 'BB_upper', 'BB_lower', 'RSI']


## 2. States
The market state at day *t* is the sigmoid of the last `WINDOW_SIZE` one-day changes of every scaled feature (0.5 = no change). The episode runner adds the agent's **position features** to it at every step.

In [4]:
# Market states. The scaler is fitted on training days only (no peeking at the test period).
scaler = StandardScaler().fit(train_df[FEATURES]) if NORMALIZE else None


def transform(df):
    return scaler.transform(df[FEATURES]) if scaler is not None else df[FEATURES].to_numpy(float)


all_states = build_states(transform(data), WINDOW_SIZE)
n_train = len(train_df)
states_train, states_test = all_states[:n_train], all_states[n_train:]
assert np.isfinite(all_states).all(), 'non-finite values in states'

STATE_SIZE = all_states.shape[1] + N_POSITION_FEATURES
print(f'Market features: {all_states.shape[1]} + position features: {N_POSITION_FEATURES} '
      f'= state size {STATE_SIZE}')

Market features: 7 + position features: 3 = state size 10


## 3. Agent

In [5]:
# Create the DQN agent. Epsilon (exploration rate) is scheduled to fall from 1.0 to its minimum over
# about 80% of training, so the agent explores early and exploits what it learned later.
total_replay_steps = EPISODES * (len(states_train) - 1) // TRAIN_EVERY
epsilon_decay = func_lib.epsilon_decay_for(total_replay_steps, AGENT_PARAMS['epsilon'], AGENT_PARAMS['epsilon_min'])
agent = Agent(state_size=STATE_SIZE, epsilon_decay=epsilon_decay, **AGENT_PARAMS)
print(f'Epsilon decay per replay step: {epsilon_decay:.6f} (reaches {AGENT_PARAMS["epsilon_min"]} '
      f'after ~{int(0.8 * total_replay_steps):,} of {total_replay_steps:,} steps)')
agent.model.summary()

Epsilon decay per replay step: 0.998755 (reaches 0.02 after ~3,140 of 3,925 steps)


## 4. Training

In [6]:
# Train for EPISODES passes over the training data, saving a model after each (model_ep1.keras, ...).
# Then chart the last episode's trades and loss, and a table of profit, trades, epsilon and loss by episode.
train_prices = train_df['Close'].to_numpy()
episode_log = []

for e in range(1, EPISODES + 1):
    result = run_episode(agent, states_train, train_prices, train=True, desc=f'Episode {e}/{EPISODES}', **RUN_PARAMS)
    print_summary(result, f'Episode {e}/{EPISODES} | epsilon = {agent.epsilon:.3f}')
    mean_loss = np.mean(result['losses']) if result['losses'] else np.nan
    episode_log.append({'Episode': e, 'Net P&L ($/share)': result['pnl'][-1], 'Closed Trades': len(result['trades']),
                        'Epsilon': agent.epsilon, 'Mean Loss': mean_loss})
    agent.model.save(f'model_ep{e}.keras')

# Charts for the final episode
plot_behavior(train_df, result, f'Training Episode {EPISODES}')
plot_losses(result['losses'], f'Episode {EPISODES} DQN Model Loss')
episode_log = pd.DataFrame(episode_log).set_index('Episode')
func_lib.stats_table_figure(episode_log.T, 'Training progress by episode', index_name='Metric', decimals=3).show()

Episode 1/5:   0%|          | 0/785 [00:00<?, ?it/s]

error: operand #1 does not dominate this use
E0000 00:00:1794277461.577523  158978 meta_optimizer.cc:967] tfg_optimizer{any(tfg-consolidate-attrs,tfg-toposort,tfg-shape-inference{graph-version=0},tfg-prepare-attrs-export)} failed: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 
W0000 00:00:1794277461.577684  158978 optimize_function_graph_utils.cc:634] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 


error: operand #0 does not dominate this use
W0000 00:00:1794277461.649642  158978 optimize_function_graph_utils.cc:634] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 
Episode 1/5:   4%|▍         | 33/785 [00:00<00:05, 140.06it/s]

Episode 1/5:   9%|▉         | 69/785 [00:00<00:03, 222.90it/s]

Episode 1/5:  14%|█▎        | 106/785 [00:00<00:02, 274.32it/s]

Episode 1/5:  19%|█▊        | 146/785 [00:00<00:02, 313.70it/s]

... (111 outputs left out)

Episode 5/5:  88%|████████▊ | 692/785 [00:02<00:00, 323.13it/s]

Episode 5/5:  92%|█████████▏| 726/785 [00:02<00:00, 325.75it/s]

Episode 5/5:  97%|█████████▋| 759/785 [00:02<00:00, 325.22it/s]

------------------------------------------------------------
Episode 5/5 | epsilon = 0.020
Closed trades: 54 | open at end (closed on last day): 3
Total Profit:  $117.73  (net of costs: $110.84)
Total Winners: $296.75
Total Losers:  -$179.02
Actions: hold 680 | buy 54 | sell 51
Max batch loss: 55.9089
------------------------------------------------------------


## 5. Out-of-sample test vs benchmarks (greedy policy, no exploration or learning)

In [7]:
# Out-of-sample test: reload the final model with exploration and learning off, trade the unseen test
# period, and compare with buy-and-hold and a random policy that use the same simulator, costs and cap.
test_prices = test_df['Close'].to_numpy()
test_agent = Agent(state_size=STATE_SIZE, model_path=FINAL_MODEL_PATH, test_mode=True)
test_result = run_episode(test_agent, states_test, test_prices, desc='Test', **RUN_PARAMS)
print_summary(test_result, 'Test period | DQN agent')
plot_behavior(test_df, test_result, 'Test Period | DQN agent')

# Benchmark policies: (step, state, valid actions) -> action
rng = np.random.default_rng(SEED)
benchmarks = {
    f'Buy & hold ({MAX_INVENTORY} shares)': lambda t, s, valid: 1 if 1 in valid else 0,
    'Random policy': lambda t, s, valid: int(rng.choice(valid)),
}
results = {'DQN agent': test_result}
for name, policy in benchmarks.items():
    results[name] = run_episode(None, states_test, test_prices, policy=policy, desc=name, **RUN_PARAMS)

func_lib.policy_comparison_figure(test_df.index, results, title='Test period: DQN agent vs benchmarks').show()
func_lib.stats_table_figure(func_lib.policy_stats_table(results), 'Test period: policy comparison',
                            index_name='Metric', pct_rows=('Win Rate', 'Days Holding')).show()

Test:   0%|          | 0/337 [00:00<?, ?it/s]

Test:  34%|███▍      | 114/337 [00:00<00:00, 1135.85it/s]

Test:  68%|██████▊   | 229/337 [00:00<00:00, 1139.27it/s]

------------------------------------------------------------
Test period | DQN agent
Closed trades: 36 | open at end (closed on last day): 3
Total Profit:  $151.80  (net of costs: $145.62)
Total Winners: $201.43
Total Losers:  -$49.64
Actions: hold 268 | buy 36 | sell 33
------------------------------------------------------------


Buy & hold (5 shares):   0%|          | 0/337 [00:00<?, ?it/s]

Random policy:   0%|          | 0/337 [00:00<?, ?it/s]

In [8]:
# Diagnostics: which actions the greedy agent chose in the test period, and a warning if it never
# sold (a sign the policy collapsed into always buying).
counts = pd.Series(test_result['actions'][:-1]).map(ACTIONS).value_counts()
print('Test actions:', counts.to_dict())
if len(test_result['trades']) == 0 or counts.get('sell', 0) == 0:
    print('WARNING: the policy never sold in the test period. Try more EPISODES, REWARD_MODE="mtm", '
          'a smaller MAX_INVENTORY, or a different SEED.')

Test actions: {'hold': 268, 'buy': 36, 'sell': 33}


## 6. Backtest on another market or period
`backtest()` runs the trained agent on any DataFrame with the same feature columns. It reuses the **training** scaler, so no information from the backtest data leaks into normalization. It is demonstrated here on the test period.

In [9]:
# Backtest helper: rerun any saved model on any data with the same features (another period or stock).
# Here it reruns the test period and builds the trade log, one row per closed trade.
def backtest(df, model_path=FINAL_MODEL_PATH):
    df = df[FEATURES].dropna()
    states = build_states(transform(df), WINDOW_SIZE)
    bt_agent = Agent(state_size=STATE_SIZE, model_path=model_path, test_mode=True)
    return df, run_episode(bt_agent, states, df['Close'].to_numpy(), desc='Backtest', **RUN_PARAMS)


bt_df, bt_result = backtest(test_df)
print_summary(bt_result, 'Backtest')
plot_behavior(bt_df, bt_result, 'Backtest')
trade_tracker = trade_report(bt_df, bt_result)
print(f'{len(trade_tracker):,} closed trades | exit reasons: {trade_tracker["Exit Reason"].value_counts().to_dict()}')
trade_tracker.head(10)

Backtest:   0%|          | 0/337 [00:00<?, ?it/s]

Backtest:  34%|███▍      | 114/337 [00:00<00:00, 1133.46it/s]

Backtest:  68%|██████▊   | 229/337 [00:00<00:00, 1138.12it/s]

------------------------------------------------------------
Backtest
Closed trades: 36 | open at end (closed on last day): 3
Total Profit:  $151.80  (net of costs: $145.62)
Total Winners: $201.43
Total Losers:  -$49.64
Actions: hold 268 | buy 36 | sell 33
------------------------------------------------------------


36 closed trades | exit reasons: {'Agent sell': 33, 'End of data': 3}


,Buy Price,Sell Price,Trade Profit,Exit Reason,Buy Timestamp,Sell Timestamp,Buy SMA20,Buy SMA50,Buy Volatility,Winner,Holding Days
0,165.709351,165.837250,0.127899,Agent sell,2023-04-27,2023-05-02,162.031786,155.088441,2.090028,True,3
1,166.958954,164.764725,-2.194229,Agent sell,2023-04-28,2023-05-03,162.391914,155.370839,2.288435,False,3
2,166.870438,163.131348,-3.739090,Agent sell,2023-05-01,2023-05-04,162.622655,155.683346,2.497110,False,3
3,170.786621,169.015472,-1.771149,Agent sell,2023-05-05,2023-05-09,163.367022,157.179167,3.126647,False,2
4,170.717728,170.037506,-0.680222,Agent sell,2023-05-08,2023-05-12,163.931327,157.706375,3.386655,False,4
5,170.776810,169.544876,-1.231934,Agent sell,2023-05-10,2023-05-15,165.133241,158.690340,3.147175,False,3
6,170.963715,169.042313,-1.921402,Agent sell,2023-05-11,2023-05-23,165.536174,159.250019,3.355843,False,8
7,169.544876,169.318207,-0.226669,Agent sell,2023-05-16,2023-05-24,166.545461,160.561736,3.444272,False,6
8,170.155746,171.885239,1.729492,Agent sell,2023-05-17,2023-09-13,166.806155,160.981473,3.513138,True,81
9,170.451355,172.674545,2.223190,Agent sell,2023-05-25,2023-09-15,169.231603,163.623014,2.517016,True,77


## 7. Post-trade analysis

In [10]:
# Post-trade analysis: win rate, profit factor, worst trade and drawdown of realized profit,
# with charts of each trade's profit and the cumulative realized P&L.
stats = trade_statistics(trade_tracker)
print_trade_statistics(stats)
trade_analysis_figure(trade_tracker, stats['realized_pnl']).show()

Closed trades:       36
Worst single trade:  -$6.46
Max drawdown (realized P&L): $37.51
Win/Loss ratio:      0.71
Profit factor:       4.06
